# T2.A2.1 — Filtros espaciales y en frecuencia sobre una misma imagen

**Curso:** Redes Neuronales Convolucionales aplicadas a la Visión Computacional  
**Clave:** PFAD-PADCC-VIC04 — TecNM Virtual  
**Tema 2:** Operaciones sobre Imágenes Digitales  
**Actividad:** T2.A2.1  
**Participante:** _(Escribe tu nombre aquí)_  
**Fecha:** _(Escribe la fecha aquí)_

---

## Objetivo

Analizar los conceptos de sistemas lineales e invariantes en el tiempo, la convolución
de señales y el filtrado de imágenes mediante ejercicios prácticos y guiados, con el
fin de comprender su aplicación en las capas convolucionales de una red neuronal
convolucional y su impacto en el procesamiento de imágenes.

---

## Instrucciones generales

1. Ejecuta cada celda en orden con `Shift + Enter`.
2. Lee los comentarios dentro del código: explican qué hace cada instrucción y por qué.
3. Completa las celdas marcadas con `# [COMPLETA]` con tus observaciones o código.
4. Usa tu propia imagen o la imagen de prueba proporcionada por el facilitador.
5. Al terminar, exporta este notebook como PDF para entrega con el nombre:  
   `T2_A2_ApellidoNombre.pdf`

**Nota para Google Colab:** sube tu imagen usando el panel de archivos (icono de carpeta
en el panel izquierdo) antes de ejecutar el Paso 2.

---

## Contenido

| Paso | Actividad | Tiempo estimado |
|------|-----------|----------------|
| 1 | Preparación del entorno | 40 min |
| 2 | Carga y visualización de la imagen | 1 hora |
| 3 | Filtros espaciales (suavizado y bordes) | 1 hora |
| 4 | Análisis comparativo de resultados | 1 hora |
| 5 | Transformada Discreta de Fourier 2D | 1 hora |
| 6 | Filtros en el dominio de la frecuencia | 1 hora |
| 7 | Comparación espacial vs frecuencia | 1 hora |
| 8 | Reflexión final y relación con CNN | 30 min |
| — | Extensión: Variante A / B / C (elige una) | — |

---
## PASO 1 — Preparación del entorno de trabajo
**Tiempo estimado: 40 minutos**

Antes de procesar imágenes, es necesario verificar que las bibliotecas requeridas
estén instaladas y disponibles. Este paso corresponde al subtema 2.1 del manual:
los sistemas LTI operan sobre datos numéricos representados como arreglos, y las
bibliotecas que se usan aquí (NumPy, OpenCV, Matplotlib) son las herramientas
con las que implementamos esos sistemas en Python.

**Bibliotecas utilizadas:**
- `numpy`: representa imágenes como arreglos numéricos y realiza operaciones matriciales.
- `opencv-python (cv2)`: carga imágenes, aplica filtros espaciales y realiza conversiones de color.
- `matplotlib`: visualiza imágenes y gráficas dentro del notebook.

In [ ]:
# ------------------------------------------------------------
# Instalacion de bibliotecas necesarias
# Ejecuta esta celda solo si trabajas en Google Colab o si
# las bibliotecas no estan instaladas en tu entorno local.
# En un entorno local con el requirements.txt del curso ya
# instalado, puedes omitir esta celda.
# ------------------------------------------------------------
# !pip install numpy matplotlib opencv-python

In [ ]:
# ------------------------------------------------------------
# Importacion de bibliotecas
# ------------------------------------------------------------

import cv2          # OpenCV: procesamiento de imagenes
import numpy as np  # NumPy: operaciones numericas y matriciales
import matplotlib.pyplot as plt  # Matplotlib: visualizacion

# ------------------------------------------------------------
# Verificacion de versiones instaladas
# Esto permite confirmar que el entorno esta correctamente
# configurado antes de continuar con la actividad.
# ------------------------------------------------------------
print("Verificacion de versiones:")
print(f"  numpy   : {np.__version__}")
print(f"  opencv  : {cv2.__version__}")

import matplotlib
print(f"  matplotlib: {matplotlib.__version__}")

print("\nEntorno listo para la actividad.")

**Registro del entorno — completa la siguiente tabla en tu reporte:**

| Biblioteca | Version instalada | Estado |
|------------|------------------|--------|
| numpy | | |
| opencv | | |
| matplotlib | | |

_(Copia los valores que imprimio la celda anterior.)_

---
## PASO 2 — Carga y visualizacion de la imagen
**Tiempo estimado: 1 hora**

Una imagen digital es una funcion discreta `f(x, y)` donde cada valor representa
la intensidad de un pixel. En este paso cargamos la imagen, la convertimos a
escala de grises (que es la representacion que usaremos para los filtros) y
analizamos sus propiedades basicas: dimensiones, tipo de dato y rango de valores.

**Por que escala de grises?**  
Los filtros espaciales y la TDF 2D que aplicaremos operan sobre una sola
componente de intensidad. Convertir a escala de grises simplifica el analisis
y permite concentrarse en los conceptos de convolución y filtrado sin la
complejidad adicional de tres canales de color.

**Nota:** OpenCV lee imágenes en formato BGR (Azul-Verde-Rojo), no RGB.
Para visualizarlas correctamente con Matplotlib hay que convertirlas a RGB.

In [ ]:
# ------------------------------------------------------------
# CONFIGURACION: nombre del archivo de imagen
# Cambia este valor por el nombre de tu imagen.
# Si usas Colab, asegurate de haberla subido primero.
# ------------------------------------------------------------
NOMBRE_IMAGEN = 'imagen_tema2.jpg'

# ------------------------------------------------------------
# Carga de la imagen con OpenCV
# cv2.imread devuelve un arreglo NumPy de forma (alto, ancho, 3)
# en el orden de canales BGR. Si no encuentra el archivo,
# devuelve None, por lo que verificamos antes de continuar.
# ------------------------------------------------------------
img_bgr = cv2.imread(NOMBRE_IMAGEN)

if img_bgr is None:
    raise FileNotFoundError(
        f"No se encontro la imagen '{NOMBRE_IMAGEN}'.\n"
        "Verifica el nombre del archivo y que este en la misma"
        " carpeta que este notebook (o subida a Colab)."
    )

# ------------------------------------------------------------
# Conversion BGR -> RGB
# Matplotlib espera canales en orden RGB, no BGR.
# Esta conversion es solo para visualizacion correcta.
# ------------------------------------------------------------
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# ------------------------------------------------------------
# Conversion a escala de grises
# La funcion cvtColor aplica la formula estandar de luminancia:
# Y = 0.299*R + 0.587*G + 0.114*B
# El resultado es un arreglo 2D de forma (alto, ancho).
# ------------------------------------------------------------
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# ------------------------------------------------------------
# Informacion basica de la imagen
# .shape devuelve (filas, columnas) para imagen en grises
# .dtype indica el tipo de dato (uint8 = enteros 0-255)
# ------------------------------------------------------------
alto, ancho = img_gray.shape
print("Propiedades de la imagen:")
print(f"  Dimensiones (alto x ancho): {alto} x {ancho} pixeles")
print(f"  Tipo de dato              : {img_gray.dtype}")
print(f"  Valor minimo de pixel     : {img_gray.min()}")
print(f"  Valor maximo de pixel     : {img_gray.max()}")
print(f"  Valor promedio de pixel   : {img_gray.mean():.2f}")

In [ ]:
# ------------------------------------------------------------
# Visualizacion: imagen en color y en escala de grises
# Se muestran lado a lado para comparar ambas representaciones.
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Imagen en color (RGB)
axes[0].imshow(img_rgb)
axes[0].set_title("Imagen en color (RGB)")
axes[0].axis('off')

# Imagen en escala de grises
# cmap='gray' indica a Matplotlib que use la paleta de grises.
axes[1].imshow(img_gray, cmap='gray')
axes[1].set_title("Imagen en escala de grises")
axes[1].axis('off')

plt.suptitle("Paso 2 — Representaciones de la imagen cargada", fontsize=12)
plt.tight_layout()
plt.savefig('paso2_carga_imagen.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()

**Registro de resultados — completa en tu reporte:**

1. Dimensiones de la imagen (alto x ancho en pixeles): ___________
2. Tipo de dato del arreglo: ___________
3. Rango de valores de intensidad (min - max): ___________
4. Describe brevemente la escena que muestra la imagen (2-3 lineas):

   _____________________________________________________________

   _____________________________________________________________

---
## PASO 3 — Filtros espaciales: suavizado y deteccion de bordes
**Tiempo estimado: 1 hora**

Los filtros espaciales son el ejemplo mas directo de un sistema LTI aplicado a
imagenes. Operan mediante convolución 2D: un kernel (matriz de coeficientes)
se desplaza pixel a pixel sobre la imagen y en cada posicion calcula una suma
ponderada de los valores vecinos.

En este paso aplicamos cuatro filtros:

**Filtros de suavizado (pasa-bajas en espacio):**
- **Promedio 3x3:** todos los coeficientes iguales (1/9). Reduce el ruido
  pero tambien borra bordes.
- **Gaussiano 5x5:** coeficientes con forma de campana. Suaviza con menos
  distorsion que el promedio porque pondera mas el pixel central.

**Filtros de deteccion de bordes (pasa-altas en espacio):**
- **Sobel:** calcula el gradiente de intensidad en X e Y por separado.
  La magnitud combinada resalta bordes en todas direcciones.
- **Laplaciano:** segunda derivada discreta. Detecta cambios abruptos de
  intensidad. Mas sensible al ruido que Sobel.

**Conexion con LTI:** todos estos filtros son lineales (combinacion lineal de
vecinos) e invariantes en el espacio (el mismo kernel se aplica en toda la
imagen). Esto los convierte en sistemas LTI 2D.

In [ ]:
# ============================================================
# FILTRO 1: Suavizado por promedio 3x3
# ============================================================
# El kernel de promedio tiene todos sus 9 coeficientes iguales
# a 1/9. La suma de todos los coeficientes es 1, lo que garantiza
# que el brillo promedio de la imagen no cambie.
#
# np.ones((3,3)) crea una matriz 3x3 de unos.
# Dividir entre 9.0 normaliza el kernel para que sume 1.
# np.float32 especifica precision de punto flotante de 32 bits.
# ------------------------------------------------------------
kernel_promedio = np.ones((3, 3), np.float32) / 9.0

# cv2.filter2D aplica una convolucion 2D generica.
# Parametros:
#   src    : imagen de entrada (escala de grises)
#   ddepth : -1 indica que la imagen de salida tiene el mismo
#            tipo de dato que la entrada (uint8)
#   kernel : matriz de coeficientes del filtro
img_promedio = cv2.filter2D(img_gray, -1, kernel_promedio)

print("Filtro promedio 3x3 aplicado.")
print(f"Kernel:\n{kernel_promedio}")

In [ ]:
# ============================================================
# FILTRO 2: Suavizado gaussiano 5x5
# ============================================================
# El filtro gaussiano usa coeficientes que siguen una distribucion
# normal bidimensional. El pixel central recibe el mayor peso;
# los vecinos mas lejanos reciben pesos menores.
#
# cv2.GaussianBlur parametros:
#   src    : imagen de entrada
#   ksize  : tamano del kernel (5x5); debe ser impar
#   sigmaX : desviacion estandar en X. Un valor mayor produce
#            mas suavizado. Con 1.0 el efecto es moderado.
# ------------------------------------------------------------
img_gauss = cv2.GaussianBlur(img_gray, (5, 5), sigmaX=1.0)

print("Filtro gaussiano 5x5 (sigma=1.0) aplicado.")

In [ ]:
# ============================================================
# FILTRO 3: Deteccion de bordes con Sobel
# ============================================================
# El filtro Sobel calcula el gradiente de intensidad en dos
# direcciones por separado:
#   sobelx: cambios horizontales (bordes verticales)
#   sobely: cambios verticales (bordes horizontales)
#
# cv2.Sobel parametros:
#   src    : imagen de entrada
#   ddepth : cv2.CV_64F usa precision de 64 bits para no perder
#            informacion de gradientes negativos (que indicarian
#            transiciones de claro a oscuro)
#   dx, dy : orden de la derivada en X e Y (1=primera derivada)
#   ksize  : tamano del kernel Sobel (3x3)
# ------------------------------------------------------------
sobelx = cv2.Sobel(img_gray, cv2.CV_64F, dx=1, dy=0, ksize=3)
sobely = cv2.Sobel(img_gray, cv2.CV_64F, dx=0, dy=1, ksize=3)

# La magnitud del gradiente combina ambas direcciones:
# magnitude = sqrt(sobelx^2 + sobely^2)
# cv2.magnitude hace este calculo de forma eficiente.
img_sobel = cv2.magnitude(sobelx, sobely)

# Convertir a uint8 para visualizacion:
# np.clip limita valores al rango [0, 255] antes de convertir.
img_sobel_vis = np.clip(img_sobel, 0, 255).astype(np.uint8)

print("Filtro Sobel aplicado.")
print(f"Magnitud del gradiente: min={img_sobel.min():.1f}, max={img_sobel.max():.1f}")

In [ ]:
# ============================================================
# FILTRO 4: Deteccion de bordes con Laplaciano
# ============================================================
# El Laplaciano es la segunda derivada discreta de la imagen.
# Detecta zonas donde la intensidad cambia rapidamente en
# cualquier direccion (no solo horizontal o vertical como Sobel).
#
# Es mas sensible al ruido que Sobel porque la segunda derivada
# amplifica variaciones pequenas. Por eso en la practica se
# suele suavizar la imagen antes de aplicarlo.
#
# cv2.Laplacian parametros:
#   ddepth : cv2.CV_64F para mantener valores negativos
#   ksize  : tamano del kernel (3x3)
# ------------------------------------------------------------
img_lap = cv2.Laplacian(img_gray, cv2.CV_64F, ksize=3)

# Para visualizar el Laplaciano se toma el valor absoluto,
# ya que tanto transiciones claras->oscuras como oscuras->claras
# son bordes igualmente validos.
img_lap_vis = np.clip(np.abs(img_lap), 0, 255).astype(np.uint8)

print("Filtro Laplaciano aplicado.")
print(f"Rango del Laplaciano: min={img_lap.min():.1f}, max={img_lap.max():.1f}")

In [ ]:
# ============================================================
# Visualizacion comparativa de todos los filtros espaciales
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

imagenes = [
    (img_gray,        "Original (grises)"),
    (img_promedio,    "Suavizado promedio 3x3"),
    (img_gauss,       "Suavizado gaussiano 5x5"),
    (img_sobel_vis,   "Bordes Sobel (magnitud)"),
    (img_lap_vis,     "Bordes Laplaciano (|valor|)"),
]

# Mostrar las 5 imagenes en las primeras 5 posiciones del grid
for idx, (imagen, titulo) in enumerate(imagenes):
    fila = idx // 3
    col  = idx % 3
    axes[fila][col].imshow(imagen, cmap='gray')
    axes[fila][col].set_title(titulo, fontsize=10)
    axes[fila][col].axis('off')

# La posicion [1][2] queda vacia; la ocultamos
axes[1][2].axis('off')

plt.suptitle("Paso 3 — Comparacion de filtros espaciales", fontsize=13)
plt.tight_layout()
plt.savefig('paso3_filtros_espaciales.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()

**Registro de resultados — completa en tu reporte:**

Para cada imagen filtrada, escribe 2-3 lineas describiendo que cambios
observas respecto a la imagen original:

| Filtro | Observacion |
|--------|-------------|
| Suavizado promedio 3x3 | |
| Suavizado gaussiano 5x5 | |
| Bordes Sobel | |
| Bordes Laplaciano | |

Guarda la figura `paso3_filtros_espaciales.png` para incluirla en tu reporte.

---
## PASO 4 — Analisis comparativo de resultados
**Tiempo estimado: 1 hora**

En este paso analizamos cuantitativamente el efecto de cada filtro. Ademas de la
comparacion visual del Paso 3, calculamos metricas que permiten describir con
precision que informacion se resalto y cual se perdio.

**Metrica utilizada — diferencia absoluta media (MAD):**  
Mide cuanto cambio en promedio cada pixel respecto a la imagen original.
Un valor alto indica que el filtro modifico significativamente la imagen.

**Metrica de bordes — densidad de bordes:**  
Para los filtros de bordes, contamos cuantos pixeles superan un umbral de
intensidad. Esto cuantifica cuantos bordes detecto cada filtro.

In [ ]:
# ============================================================
# Calculo de metricas cuantitativas por filtro
# ============================================================

# Diferencia absoluta media respecto a la imagen original
# np.mean(np.abs(A - B)) promedia la magnitud de todos los
# cambios pixel a pixel entre la imagen filtrada y la original.
mad_promedio = np.mean(np.abs(img_promedio.astype(float) - img_gray.astype(float)))
mad_gauss    = np.mean(np.abs(img_gauss.astype(float)    - img_gray.astype(float)))

# Para los filtros de bordes, la metrica relevante es la
# densidad de bordes: porcentaje de pixeles con magnitud > umbral
UMBRAL_BORDE = 30  # valor de intensidad considerado "borde"
densidad_sobel = np.mean(img_sobel_vis > UMBRAL_BORDE) * 100
densidad_lap   = np.mean(img_lap_vis   > UMBRAL_BORDE) * 100

# Estadisticas de intensidad de cada imagen filtrada
print("=" * 55)
print("Analisis cuantitativo de filtros espaciales")
print("=" * 55)

print("\nFiltros de suavizado — diferencia respecto al original:")
print(f"  Promedio 3x3    : MAD = {mad_promedio:.3f} niveles de gris")
print(f"  Gaussiano 5x5   : MAD = {mad_gauss:.3f} niveles de gris")

print("\nFiltros de bordes — densidad de bordes detectados:")
print(f"  Sobel           : {densidad_sobel:.2f}% de pixeles son borde")
print(f"  Laplaciano      : {densidad_lap:.2f}% de pixeles son borde")

print("\nEstadisticas de intensidad de imagenes filtradas:")
for nombre, img in [("Original", img_gray),
                    ("Promedio", img_promedio),
                    ("Gaussiano", img_gauss),
                    ("Sobel", img_sobel_vis),
                    ("Laplaciano", img_lap_vis)]:
    print(f"  {nombre:<12}: media={img.mean():.1f}, std={img.std():.1f}")

In [ ]:
# ============================================================
# Visualizacion: histogramas de intensidad
# ============================================================
# El histograma muestra la distribucion de valores de gris en
# la imagen. Comparar histogramas entre la original y la filtrada
# permite ver como cada filtro redistribuye la informacion.
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Histograma de suavizado
axes[0].hist(img_gray.ravel(),     bins=64, alpha=0.6, color='black',  label='Original')
axes[0].hist(img_promedio.ravel(), bins=64, alpha=0.6, color='blue',   label='Promedio 3x3')
axes[0].hist(img_gauss.ravel(),    bins=64, alpha=0.6, color='orange', label='Gaussiano 5x5')
axes[0].set_title("Histograma: suavizado")
axes[0].set_xlabel("Intensidad")
axes[0].set_ylabel("Numero de pixeles")
axes[0].legend()

# Histograma de bordes (Sobel)
axes[1].hist(img_sobel_vis.ravel(), bins=64, color='red', alpha=0.8)
axes[1].axvline(x=UMBRAL_BORDE, color='black', linestyle='--', label=f'Umbral={UMBRAL_BORDE}')
axes[1].set_title("Histograma: Sobel")
axes[1].set_xlabel("Intensidad (magnitud del gradiente)")
axes[1].legend()

# Histograma de bordes (Laplaciano)
axes[2].hist(img_lap_vis.ravel(), bins=64, color='green', alpha=0.8)
axes[2].axvline(x=UMBRAL_BORDE, color='black', linestyle='--', label=f'Umbral={UMBRAL_BORDE}')
axes[2].set_title("Histograma: Laplaciano")
axes[2].set_xlabel("Intensidad (|Laplaciano|)")
axes[2].legend()

plt.suptitle("Paso 4 — Histogramas de intensidad por filtro", fontsize=12)
plt.tight_layout()
plt.savefig('paso4_histogramas.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()

**Preguntas de analisis — responde en tu reporte (2-3 lineas cada una):**

1. Segun la metrica MAD, cual filtro de suavizado modifico mas la imagen?
   Por que crees que ocurre eso?

2. Sobel y Laplaciano detectaron porcentajes distintos de bordes.
   Cual detecto mas? Que puede explicar esa diferencia?

3. Observando los histogramas, que le ocurre a la distribucion de intensidades
   despues del suavizado en comparacion con la imagen original?

---
## PASO 5 — Transformada Discreta de Fourier 2D
**Tiempo estimado: 1 hora**

Hasta ahora hemos trabajado en el dominio espacial: operamos directamente sobre
los valores de intensidad de los pixeles. La Transformada Discreta de Fourier
(TDF) nos permite pasar al dominio de la frecuencia, donde la imagen se
representa como combinacion de patrones senoidales de distintas frecuencias.

**Interpretacion de frecuencias en una imagen:**
- **Frecuencias bajas** (centro del espectro): variaciones suaves de intensidad,
  como fondos uniformes y transiciones graduales.
- **Frecuencias altas** (bordes del espectro): variaciones abruptas, como bordes,
  detalles finos y ruido.

**Por que usar la TDF?**  
Ciertos filtros son mas faciles de disenar e interpretar en frecuencia.
Ademas, existe una relacion fundamental entre ambos dominios:
convolucion en el espacio es equivalente a multiplicacion en frecuencia.
Esta propiedad es la base matematica de las capas convolucionales en CNNs.

In [ ]:
# ============================================================
# Calculo de la Transformada Discreta de Fourier 2D
# ============================================================

# np.fft.fft2 calcula la TDF bidimensional de la imagen.
# El resultado es un arreglo de numeros complejos del mismo
# tamano que la imagen de entrada.
f = np.fft.fft2(img_gray)

# np.fft.fftshift desplaza el componente de frecuencia cero
# (DC) al centro del arreglo. Sin este paso, las frecuencias
# bajas estarian en las esquinas, lo que dificulta la
# interpretacion visual y la construccion de mascaras.
fshift = np.fft.fftshift(f)

# El espectro de magnitud indica la intensidad de cada
# componente de frecuencia. Se aplica logaritmo para comprimir
# el rango dinamico (los valores abarcan varios ordenes de
# magnitud) y hacerlo visible en pantalla.
# Se suma 1e-8 para evitar log(0) cuando la magnitud es cero.
magnitude_spectrum = 20 * np.log(np.abs(fshift) + 1e-8)

print("TDF 2D calculada correctamente.")
print(f"Tamano del espectro : {fshift.shape}")
print(f"Rango del espectro  : [{magnitude_spectrum.min():.1f}, {magnitude_spectrum.max():.1f}] dB")

In [ ]:
# ============================================================
# Visualizacion del espectro de magnitud
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Imagen original en escala de grises
axes[0].imshow(img_gray, cmap='gray')
axes[0].set_title("Imagen original (grises)")
axes[0].axis('off')

# Espectro de magnitud
# El punto brillante en el centro corresponde a la componente
# de frecuencia cero (valor medio de la imagen).
# La energia decrece hacia los bordes, que representan
# frecuencias mas altas.
axes[1].imshow(magnitude_spectrum, cmap='gray')
axes[1].set_title("Espectro de magnitud (TDF 2D)")
axes[1].axis('off')

plt.suptitle("Paso 5 — Representacion de la imagen en frecuencia", fontsize=12)
plt.tight_layout()
plt.savefig('paso5_espectro_fourier.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()

**Preguntas de analisis — responde en tu reporte:**

1. Donde se concentra la mayor parte de la energia en el espectro de magnitud?
   Que tipo de informacion representa esa zona?

2. Si la imagen tuviera bordes muy definidos (por ejemplo, texto negro sobre
   fondo blanco), esperarias ver mas energia en frecuencias altas o bajas?
   Justifica tu respuesta.

3. El espectro de tu imagen muestra algun patron particular (lineas, simetrias)?
   Describe lo que observas.

Guarda la figura `paso5_espectro_fourier.png` para incluirla en tu reporte.

---
## PASO 6 — Aplicacion de filtros en el dominio de la frecuencia
**Tiempo estimado: 1 hora**

En el dominio de la frecuencia, filtrar una imagen es conceptualmente simple:
se multiplica el espectro por una mascara que pone a cero las frecuencias que
se desean eliminar. Despues se aplica la transformada inversa para regresar al
dominio espacial.

**Filtro pasa-bajas:** conserva solo las frecuencias bajas (circulo en el centro
del espectro). El resultado equivale a suavizar la imagen, igual que el filtro
gaussiano del Paso 3.

**Filtro pasa-altas:** conserva solo las frecuencias altas (todo el espectro
excepto el centro). El resultado resalta bordes y detalles, similar a Sobel
o Laplaciano del Paso 3.

Esta equivalencia entre dominios es la propiedad fundamental que conecta
los filtros clasicos con el funcionamiento de las capas convolucionales.

In [ ]:
# ============================================================
# Construccion de mascaras pasa-bajas y pasa-altas
# ============================================================

# Dimensiones de la imagen
filas, cols = img_gray.shape

# Centro del espectro (despues de fftshift)
fila_centro = filas // 2
col_centro  = cols  // 2

# Radio del circulo de la mascara pasa-bajas
# Se usa 1/8 de la dimension menor de la imagen como radio.
# Un radio mayor deja pasar mas frecuencias y suaviza menos.
radio = min(filas, cols) // 8

# Mascara pasa-bajas: circulo de unos en el centro
# np.zeros crea la mascara inicialmente con todo ceros.
# cv2.circle dibuja un circulo relleno de 1s centrado en
# (col_centro, fila_centro) con el radio definido.
mask_low = np.zeros((filas, cols), dtype=np.uint8)
cv2.circle(mask_low, center=(col_centro, fila_centro),
           radius=radio, color=1, thickness=-1)

# Mascara pasa-altas: complemento de la pasa-bajas
# Donde la pasa-bajas tiene 1 (frecuencias bajas), la
# pasa-altas tiene 0, y viceversa.
mask_high = 1 - mask_low

print(f"Dimensiones de la imagen : {filas} x {cols} pixeles")
print(f"Radio de la mascara      : {radio} pixeles en el espectro")
print(f"Frecuencias en pasa-bajas: {mask_low.sum()} coeficientes")
print(f"Frecuencias en pasa-altas: {mask_high.sum()} coeficientes")

In [ ]:
# ============================================================
# Aplicacion de los filtros en frecuencia y reconstruccion
# ============================================================

# Multiplicacion del espectro por la mascara
# fshift ya es el espectro centrado calculado en el Paso 5.
# Multiplicar por la mascara anula (pone a cero) las
# frecuencias que la mascara no deja pasar.
f_low  = fshift * mask_low
f_high = fshift * mask_high

# Transformada inversa para regresar al dominio espacial
# np.fft.ifftshift deshace el centrado aplicado con fftshift.
# np.fft.ifft2 calcula la transformada inversa.
# np.abs toma la magnitud del resultado complejo (la parte
# imaginaria deberia ser casi cero para imagenes reales).
img_low  = np.abs(np.fft.ifft2(np.fft.ifftshift(f_low)))
img_high = np.abs(np.fft.ifft2(np.fft.ifftshift(f_high)))

# Normalizar al rango [0, 255] para visualizacion
def normalizar_uint8(img):
    """Escala una imagen flotante al rango [0, 255] en uint8."""
    mn, mx = img.min(), img.max()
    if mx - mn < 1e-8:  # imagen constante
        return np.zeros_like(img, dtype=np.uint8)
    return ((img - mn) / (mx - mn) * 255).astype(np.uint8)

img_low_vis  = normalizar_uint8(img_low)
img_high_vis = normalizar_uint8(img_high)

print("Filtros en frecuencia aplicados.")
print(f"  Pasa-bajas: rango de intensidad [{img_low.min():.1f}, {img_low.max():.1f}]")
print(f"  Pasa-altas: rango de intensidad [{img_high.min():.1f}, {img_high.max():.1f}]")

In [ ]:
# ============================================================
# Visualizacion: mascaras, espectros filtrados e imagenes
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

# Fila superior: mascaras y espectros filtrados
axes[0][0].imshow(mask_low, cmap='gray')
axes[0][0].set_title("Mascara pasa-bajas")
axes[0][0].axis('off')

axes[0][1].imshow(mask_high, cmap='gray')
axes[0][1].set_title("Mascara pasa-altas")
axes[0][1].axis('off')

axes[0][2].imshow(magnitude_spectrum, cmap='gray')
axes[0][2].set_title("Espectro original")
axes[0][2].axis('off')

# Fila inferior: imagenes reconstruidas
axes[1][0].imshow(img_gray, cmap='gray')
axes[1][0].set_title("Original (grises)")
axes[1][0].axis('off')

axes[1][1].imshow(img_low_vis, cmap='gray')
axes[1][1].set_title("Pasa-bajas (suavizado)")
axes[1][1].axis('off')

axes[1][2].imshow(img_high_vis, cmap='gray')
axes[1][2].set_title("Pasa-altas (bordes/detalles)")
axes[1][2].axis('off')

plt.suptitle("Paso 6 — Filtrado en el dominio de la frecuencia", fontsize=13)
plt.tight_layout()
plt.savefig('paso6_filtros_frecuencia.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()

**Registro de resultados — completa en tu reporte:**

| Filtro en frecuencia | Que tipo de informacion conserva | Efecto visual observado |
|----------------------|----------------------------------|------------------------|
| Pasa-bajas | | |
| Pasa-altas | | |

Guarda la figura `paso6_filtros_frecuencia.png` para incluirla en tu reporte.

---
## PASO 7 — Comparacion: dominio espacial vs dominio de la frecuencia
**Tiempo estimado: 1 hora**

Ahora que tenemos resultados de ambos dominios, podemos compararlos directamente.
La propiedad teorica fundamental dice:

> Convolucion en el dominio espacial es equivalente a multiplicacion
> punto a punto en el dominio de la frecuencia.

Esto implica que:
- El filtro **gaussiano** (suavizado espacial) es equivalente a un **pasa-bajas**.
- El filtro **Sobel / Laplaciano** (deteccion de bordes) es equivalente a un **pasa-altas**.

En este paso visualizamos esa equivalencia lado a lado.

In [ ]:
# ============================================================
# Comparacion lado a lado: espacial vs frecuencia
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

# Fila 1: equivalentes de suavizado
axes[0][0].imshow(img_gray, cmap='gray')
axes[0][0].set_title("Original")
axes[0][0].axis('off')

axes[0][1].imshow(img_gauss, cmap='gray')
axes[0][1].set_title("Gaussiano (dominio espacial)")
axes[0][1].axis('off')

axes[0][2].imshow(img_low_vis, cmap='gray')
axes[0][2].set_title("Pasa-bajas (dominio frecuencia)")
axes[0][2].axis('off')

# Fila 2: equivalentes de deteccion de bordes
axes[1][0].imshow(img_gray, cmap='gray')
axes[1][0].set_title("Original")
axes[1][0].axis('off')

axes[1][1].imshow(img_sobel_vis, cmap='gray')
axes[1][1].set_title("Sobel (dominio espacial)")
axes[1][1].axis('off')

axes[1][2].imshow(img_high_vis, cmap='gray')
axes[1][2].set_title("Pasa-altas (dominio frecuencia)")
axes[1][2].axis('off')

plt.suptitle(
    "Paso 7 — Equivalencia entre dominios\n"
    "Fila 1: suavizado | Fila 2: deteccion de bordes",
    fontsize=12
)
plt.tight_layout()
plt.savefig('paso7_comparacion_dominios.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# ============================================================
# Medicion cuantitativa de la similitud entre dominios
# ============================================================
# Calculamos la correlacion de Pearson entre los resultados
# de ambos dominios para verificar que son similares.
# Una correlacion cercana a 1.0 indica alta similitud.
# ------------------------------------------------------------

def correlacion(a, b):
    """Calcula el coeficiente de correlacion de Pearson entre dos imagenes."""
    a_f = a.astype(float).ravel()
    b_f = b.astype(float).ravel()
    return np.corrcoef(a_f, b_f)[0, 1]

corr_suavizado = correlacion(img_gauss, img_low_vis)
corr_bordes    = correlacion(img_sobel_vis, img_high_vis)

print("Similitud entre dominios espacial y frecuencia:")
print(f"  Gaussiano vs Pasa-bajas   : r = {corr_suavizado:.4f}")
print(f"  Sobel vs Pasa-altas       : r = {corr_bordes:.4f}")
print()
print("Nota: valores cercanos a 1.0 confirman la equivalencia teorica")
print("entre la convolucion espacial y el filtrado en frecuencia.")

**Reflexion guiada — responde en tu reporte (10-15 lineas en total):**

Considerando los resultados visuales y la correlacion calculada:

1. El filtro gaussiano (espacial) y el pasa-bajas (frecuencia) producen resultados
   visualmente similares. Que informacion se atenua en ambos casos y cual se conserva?

2. El filtro Sobel (espacial) y el pasa-altas (frecuencia) muestran resultados
   parecidos. Que diferencias sutiles puedes notar entre ambas imagenes?

3. Con base en la propiedad de equivalencia entre dominios, explica por que las
   capas convolucionales de una CNN (que aplican kernels pequeños en el espacio)
   pueden interpretarse como filtros en el dominio de la frecuencia.

4. Que ventaja practica tiene disenar un filtro en frecuencia respecto a disenar
   un kernel espacial manualmente?

Guarda la figura `paso7_comparacion_dominios.png` para incluirla en tu reporte.

---
## PASO 7B — Ejercicio 2: Diseno de kernels propios
**Tiempo estimado: 1 hora**

En este ejercicio disenamos manualmente kernels 3x3 y los comparamos con
los filtros clasicos aplicados en el Paso 3. El objetivo es comprender la
relacion entre la forma de los coeficientes y el efecto visual del filtro.

**Kernel de suavizado:** los coeficientes deben ser no negativos y sumar 1,
para que el brillo promedio de la imagen no cambie tras el filtrado.

**Kernel de bordes horizontales:** los coeficientes tienen valores positivos
en una fila y negativos en la opuesta, con suma total igual a cero. Esto
produce respuesta nula en zonas uniformes y maxima en transiciones de intensidad.

**Preguntas guia:**
- Que suma deben tener los coeficientes para un filtro de suavizado?
- Que forma tienden a tener los filtros detectores de bordes (positivos y negativos)?
- Como se compara tu kernel con Sobel, Prewitt o Laplaciano?

In [ ]:
# ============================================================
# PASO 7B — Diseno de kernel de suavizado propio
# ============================================================
# Un kernel de suavizado debe tener coeficientes no negativos
# cuya suma sea 1 (para que el brillo promedio no cambie).
# Este ejemplo pondera mas el pixel central, similar al gaussiano.
# ------------------------------------------------------------
kernel_suavizado_propio = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1]
], dtype=np.float32) / 16.0   # suma total = 16, al dividir queda en 1

# ============================================================
# Diseno de kernel de bordes horizontales propio
# ============================================================
# Los coeficientes positivos detectan la transicion oscuro->claro
# y los negativos la transicion claro->oscuro en direccion vertical.
# La suma total es 0: respuesta nula en zonas de intensidad uniforme.
# Este kernel es equivalente al operador Sobel en Y.
# ------------------------------------------------------------
kernel_bordes_h_propio = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
], dtype=np.float32)

# Aplicar kernel de suavizado propio
# cv2.filter2D aplica la convolucion 2D con el kernel diseniado.
img_suavizado_propio = cv2.filter2D(img_gray, -1, kernel_suavizado_propio)

# Aplicar kernel de bordes horizontales propio
# Se usa CV_64F para no perder informacion de gradientes negativos.
img_bordes_h = cv2.filter2D(img_gray.astype(np.float64),
                             cv2.CV_64F, kernel_bordes_h_propio)
img_bordes_h_vis = np.clip(np.abs(img_bordes_h), 0, 255).astype(np.uint8)

print("Kernels disenados:")
print(f"Suavizado propio (suma={kernel_suavizado_propio.sum():.1f}):")
print(kernel_suavizado_propio * 16, "/ 16")
print(f"\nBordes horizontales (suma={kernel_bordes_h_propio.sum():.1f}):")
print(kernel_bordes_h_propio)

In [ ]:
# ============================================================
# Visualizacion comparativa: kernels propios vs referencia
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

# Fila 1: comparacion de suavizado
axes[0][0].imshow(img_gray, cmap='gray')
axes[0][0].set_title("Original")
axes[0][0].axis('off')

axes[0][1].imshow(img_suavizado_propio, cmap='gray')
axes[0][1].set_title("Suavizado propio (kernel disenado)")
axes[0][1].axis('off')

axes[0][2].imshow(img_gauss, cmap='gray')
axes[0][2].set_title("Gaussiano 5x5 (referencia)")
axes[0][2].axis('off')

# Fila 2: comparacion de bordes
axes[1][0].imshow(img_gray, cmap='gray')
axes[1][0].set_title("Original")
axes[1][0].axis('off')

axes[1][1].imshow(img_bordes_h_vis, cmap='gray')
axes[1][1].set_title("Bordes horizontales propios")
axes[1][1].axis('off')

axes[1][2].imshow(img_sobel_vis, cmap='gray')
axes[1][2].set_title("Sobel (referencia)")
axes[1][2].axis('off')

plt.suptitle("Paso 7B — Kernels propios vs filtros de referencia", fontsize=12)
plt.tight_layout()
plt.savefig('paso7b_kernels_propios.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()

**Registro de resultados — completa en tu reporte:**

| Kernel | Suma de coeficientes | Efecto observado | Similitud con filtro de referencia |
|--------|---------------------|------------------|-----------------------------------|
| Suavizado propio | | | |
| Bordes horizontales | | | |

Guarda la figura `paso7b_kernels_propios.png` para incluirla en tu reporte.

---
## PASO 7C — Ejercicio 3: Comparacion de filtros clasicos y filtros CNN
**Tiempo estimado: 1 hora**

En este ejercicio visualizamos los filtros aprendidos por la primera capa
convolucional de una CNN preentrenada (VGG16) y los comparamos con los
filtros clasicos que disenamos en los pasos anteriores.

**Objetivo:** vincular la teoria de convolución con el funcionamiento real
de las redes neuronales convolucionales.

**Preguntas guia:**
- En que se parecen visualmente los kernels aprendidos y los disenados manualmente?
- Que ventajas tiene que los filtros sean aprendidos automaticamente y no fijados a priori?

**Nota:** esta seccion requiere TensorFlow. En Google Colab ya viene instalado.
En entorno local usa: `pip install tensorflow`

In [ ]:
# ============================================================
# PASO 7C — Parte A: Aplicar filtro Sobel clasico
# ============================================================
# Recordamos el resultado del Sobel del Paso 3 para tenerlo
# como referencia de comparacion con los filtros CNN.
# img_sobel_vis ya fue calculado en el Paso 3.
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(img_gray, cmap='gray')
axes[0].set_title("Imagen original")
axes[0].axis('off')

axes[1].imshow(img_sobel_vis, cmap='gray')
axes[1].set_title("Sobel clasico (magnitud del gradiente)")
axes[1].axis('off')

plt.suptitle("Paso 7C — Parte A: Filtro Sobel clasico (referencia)", fontsize=12)
plt.tight_layout()
plt.savefig('paso7c_sobel_referencia.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()

print("Sobel (Paso 3) mostrado como referencia para comparacion.")

In [ ]:
# ============================================================
# PASO 7C — Parte B: Visualizacion de filtros CNN preentrenados
# ============================================================
# Cargamos VGG16 con pesos de ImageNet y visualizamos los
# filtros de su primera capa convolucional.
#
# Los pesos tienen forma (3, 3, 3, 64):
#   3x3   : tamano del kernel
#   3     : canales de entrada (RGB)
#   64    : numero de filtros en la primera capa
# ------------------------------------------------------------

try:
    import tensorflow as tf
    from tensorflow.keras.applications import VGG16

    print(f"TensorFlow version: {tf.__version__}")

    # Cargar VGG16 con pesos preentrenados en ImageNet
    # include_top=False omite las capas densas finales
    modelo = VGG16(weights='imagenet', include_top=False)
    print(f"Modelo cargado: {modelo.name}")

    # Obtener pesos de la primera capa convolucional (layers[1])
    # get_weights() devuelve [pesos_kernel, sesgos]
    primera_capa = modelo.layers[1]
    pesos, _ = primera_capa.get_weights()
    print(f"Nombre de la capa        : {primera_capa.name}")
    print(f"Forma de los pesos       : {pesos.shape}")
    print(f"  (alto, ancho, canales_entrada, n_filtros)")

    # Normalizar pesos al rango [0, 1] para visualizacion
    pesos_norm = (pesos - pesos.min()) / (pesos.max() - pesos.min() + 1e-8)

    # Visualizar los primeros 16 filtros como promedio de los 3 canales RGB
    # Esto permite comparar visualmente con kernels en escala de grises
    fig, axes = plt.subplots(4, 4, figsize=(9, 9))

    for i in range(16):
        fila = i // 4
        col  = i % 4
        # Promedio de los 3 canales para visualizacion en escala de grises
        filtro_gris = pesos_norm[:, :, :, i].mean(axis=2)
        axes[fila][col].imshow(filtro_gris, cmap='gray',
                               interpolation='nearest', vmin=0, vmax=1)
        axes[fila][col].set_title(f"Filtro {i+1}", fontsize=8)
        axes[fila][col].axis('off')

    plt.suptitle(
        "Paso 7C — Primeros 16 filtros aprendidos por VGG16\n"
        "(primera capa convolucional, promedio de canales RGB)",
        fontsize=11
    )
    plt.tight_layout()
    plt.savefig('paso7c_filtros_vgg16.png', dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()

except ImportError:
    print("TensorFlow no esta instalado en este entorno.")
    print("Opciones para ejecutar esta seccion:")
    print("  - En Google Colab: ya viene instalado, solo ejecuta la celda.")
    print("  - En entorno local: pip install tensorflow")

**Preguntas de analisis — responde en tu reporte:**

1. Observa los 16 filtros de VGG16. Identificas algun patron visual similar
   a los filtros clasicos que implementaste (Sobel, Gaussiano, Laplaciano)?
   Describe al menos dos similitudes.

2. En que se diferencian los filtros aprendidos por la CNN de los filtros
   que disenaste manualmente en el Paso 7B?

3. Que ventaja tiene que una CNN aprenda sus propios filtros durante el
   entrenamiento en lugar de usar filtros fijados manualmente?

Guarda las figuras `paso7c_sobel_referencia.png` y `paso7c_filtros_vgg16.png`
para incluirlas en tu reporte.

---
## PASO 8 — Reflexion final y relacion con CNN
**Tiempo estimado: 30 minutos**

En este ultimo paso sintetizas lo aprendido conectando los filtros clasicos
con el funcionamiento de las redes neuronales convolucionales.

**Punto de conexion clave:**  
Una capa convolucional en una CNN aplica exactamente la misma operacion que
hemos realizado en este notebook: un kernel pequeño se desplaza sobre la imagen
y calcula sumas ponderadas (convolución 2D). La diferencia es que en una CNN
los coeficientes del kernel no se disenan manualmente, sino que se aprenden
automaticamente durante el entrenamiento para minimizar el error de clasificacion.

Las primeras capas de una CNN aprenden filtros que se parecen a los que
acabas de implementar: detectores de bordes, detectores de texturas, filtros
de suavizado. Las capas mas profundas combinan estas respuestas para detectar
patrones mas complejos (partes de objetos, formas, etc.).

In [ ]:
# ============================================================
# Resumen visual final: todos los resultados de la actividad
# ============================================================
fig, axes = plt.subplots(3, 3, figsize=(14, 13))

resultados = [
    (img_gray,        "1. Original"),
    (img_promedio,    "2. Promedio 3x3"),
    (img_gauss,       "3. Gaussiano 5x5"),
    (img_sobel_vis,   "4. Sobel"),
    (img_lap_vis,     "5. Laplaciano"),
    (magnitude_spectrum.astype(np.uint8),
                      "6. Espectro TDF"),
    (img_low_vis,     "7. Pasa-bajas"),
    (img_high_vis,    "8. Pasa-altas"),
]

for idx, (imagen, titulo) in enumerate(resultados):
    fila = idx // 3
    col  = idx % 3
    axes[fila][col].imshow(imagen, cmap='gray')
    axes[fila][col].set_title(titulo, fontsize=10)
    axes[fila][col].axis('off')

# Casilla extra vacia
axes[2][2].axis('off')

plt.suptitle(
    "Paso 8 — Resumen de la actividad T2.A2.1\n"
    "Filtros espaciales y en frecuencia",
    fontsize=13
)
plt.tight_layout()
plt.savefig('paso8_resumen_actividad.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()

print("Actividad T2.A2.1 completada.")
print("Archivos generados:")
for archivo in ['paso2_carga_imagen.png',
                'paso3_filtros_espaciales.png',
                'paso4_histogramas.png',
                'paso5_espectro_fourier.png',
                'paso6_filtros_frecuencia.png',
                'paso7_comparacion_dominios.png',
                'paso8_resumen_actividad.png']:
    print(f"  {archivo}")

**Reflexion final — escribe minimo media cuartilla en tu reporte:**

Tu reflexion debe abordar al menos los siguientes dos puntos:

**a) Importancia de entender LTI, convolucion y TDF para la vision por computadora:**  
Reflexiona sobre como los conceptos matematicos de este tema (linealidad,
invarianza, convolucion, dominio de la frecuencia) son la base de las
tecnicas modernas de procesamiento de imagenes y redes neuronales.

**b) Relacion entre filtros clasicos y filtros en CNN:**  
Explica con tus propias palabras en que se parecen y en que se diferencian
los filtros que implementaste manualmente (Sobel, Gaussiano, etc.) y los
filtros convolucionales que aprende una CNN durante el entrenamiento.
Menciona el rol del pooling y las funciones de activacion no lineales.

---
## Lista de archivos generados por esta actividad

Verifica que los siguientes archivos fueron creados antes de exportar el reporte:

| Archivo | Paso |
|---------|------|
| `paso2_carga_imagen.png` | 2 |
| `paso3_filtros_espaciales.png` | 3 |
| `paso4_histogramas.png` | 4 |
| `paso5_espectro_fourier.png` | 5 |
| `paso6_filtros_frecuencia.png` | 6 |
| `paso7_comparacion_dominios.png` | 7 |
| `paso7b_kernels_propios.png` | 7B |
| `paso7c_sobel_referencia.png` | 7C |
| `paso7c_filtros_vgg16.png` | 7C |
| `paso8_resumen_actividad.png` | 8 |

**Nombre del archivo de entrega:** `T2_A2_ApellidoNombre.pdf`

---
*PFAD-PADCC-VIC04 — TecNM Virtual — 2025*